In [ ]:
# ============================================================
# PROJECT 9: HEART DISEASE
# RANDOM FOREST ENSEMBLE
# ============================================================

# ------------------------------------------------------------
# 1. IMPORT LIBRARIES
# ------------------------------------------------------------

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import plot_tree

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve
)

print("Libraries imported successfully!")


# ------------------------------------------------------------
# 2. UPLOAD DATASET
# ------------------------------------------------------------

uploaded = files.upload()

file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name)

print("\nDataset loaded successfully!")
print("File:", file_name)


# ------------------------------------------------------------
# 3. BASIC DATA EXPLORATION
# ------------------------------------------------------------

print("\n========== FIRST 5 ROWS ==========")
display(df.head())

print("\n========== DATASET SHAPE ==========")
print(df.shape)

print("\n========== COLUMN NAMES ==========")
print(df.columns.tolist())

print("\n========== DATA TYPES ==========")
print(df.dtypes)

print("\n========== DATASET INFORMATION ==========")
df.info()

print("\n========== STATISTICAL SUMMARY ==========")
display(df.describe(include="all").T)

print("\n========== MISSING VALUES ==========")
print(df.isnull().sum())

print("\n========== DUPLICATE ROWS ==========")
print(df.duplicated().sum())


# ------------------------------------------------------------
# 4. CLEAN COLUMN NAMES
# ------------------------------------------------------------

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

print("\nCleaned column names:")
print(df.columns.tolist())


# ------------------------------------------------------------
# 5. HANDLE '?' VALUES
# ------------------------------------------------------------

df = df.replace("?", np.nan)

print("\nMissing values after replacing '?':")
print(df.isnull().sum())


# ------------------------------------------------------------
# 6. REMOVE DUPLICATES
# ------------------------------------------------------------

before = len(df)

df = df.drop_duplicates()

after = len(df)

print("\nDuplicates removed:", before - after)
print("New dataset shape:", df.shape)


# ------------------------------------------------------------
# 7. FIND TARGET COLUMN
# ------------------------------------------------------------

possible_targets = [
    "target",
    "heartdisease",
    "heart_disease",
    "heart_disease_present",
    "output",
    "condition",
    "num",
    "diagnosis"
]

TARGET = None

for column in possible_targets:

    if column in df.columns:
        TARGET = column
        break


# If target is not automatically detected,
# change the following line.

if TARGET is None:
    TARGET = "target"


print("\nTarget column:", TARGET)

if TARGET not in df.columns:

    raise ValueError(
        f"Target column '{TARGET}' was not found.\n\n"
        f"Available columns:\n{df.columns.tolist()}\n\n"
        "Please change TARGET to your actual target column."
    )


# ------------------------------------------------------------
# 8. CONVERT NUMERIC COLUMNS
# ------------------------------------------------------------

for column in df.columns:

    if column != TARGET:

        converted = pd.to_numeric(
            df[column],
            errors="coerce"
        )

        if converted.notna().sum() >= (
            df[column].notna().sum() * 0.8
        ):

            df[column] = converted


# ------------------------------------------------------------
# 9. TARGET DISTRIBUTION
# ------------------------------------------------------------

print("\n========== TARGET DISTRIBUTION ==========")

print(
    df[TARGET].value_counts()
)

print("\nTarget percentages:")

print(
    df[TARGET]
    .value_counts(normalize=True)
    .mul(100)
)


# ------------------------------------------------------------
# 10. TARGET VISUALIZATION
# ------------------------------------------------------------

plt.figure(figsize=(8, 5))

sns.countplot(
    data=df,
    x=TARGET
)

plt.title(
    "Heart Disease Target Distribution"
)

plt.xlabel(
    "Heart Disease"
)

plt.ylabel(
    "Number of Patients"
)

plt.show()


# ------------------------------------------------------------
# 11. SEPARATE FEATURES AND TARGET
# ------------------------------------------------------------

X = df.drop(
    columns=[TARGET]
)

y = df[TARGET]

print("\nFeatures shape:", X.shape)
print("Target shape:", y.shape)


# ------------------------------------------------------------
# 12. IDENTIFY NUMERIC FEATURES
# ------------------------------------------------------------

numeric_features = X.select_dtypes(
    include=[
        "int64",
        "float64",
        "int32",
        "float32"
    ]
).columns.tolist()


# ------------------------------------------------------------
# 13. IDENTIFY CATEGORICAL FEATURES
# ------------------------------------------------------------

categorical_features = X.select_dtypes(
    include=[
        "object",
        "category",
        "bool"
    ]
).columns.tolist()


print("\n========== NUMERIC FEATURES ==========")
print(numeric_features)

print("\n========== CATEGORICAL FEATURES ==========")
print(categorical_features)


# ------------------------------------------------------------
# 14. EXPLORATORY DATA ANALYSIS
# ------------------------------------------------------------

if len(numeric_features) > 0:

    X[numeric_features].hist(
        figsize=(16, 12),
        bins=20
    )

    plt.suptitle(
        "Heart Disease Feature Distributions",
        fontsize=16
    )

    plt.tight_layout()

    plt.show()


# ------------------------------------------------------------
# 15. CORRELATION HEATMAP
# ------------------------------------------------------------

if len(numeric_features) > 1:

    plt.figure(figsize=(12, 9))

    sns.heatmap(
        df[numeric_features].corr(),
        annot=True,
        fmt=".2f",
        cmap="coolwarm"
    )

    plt.title(
        "Feature Correlation Heatmap"
    )

    plt.show()


# ------------------------------------------------------------
# 16. TRAIN-TEST SPLIT
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\n========== TRAIN-TEST SPLIT ==========")

print(
    "Training samples:",
    X_train.shape[0]
)

print(
    "Testing samples:",
    X_test.shape[0]
)


# ------------------------------------------------------------
# 17. NUMERIC PREPROCESSING
# ------------------------------------------------------------

numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)


# ------------------------------------------------------------
# 18. CATEGORICAL PREPROCESSING
# ------------------------------------------------------------

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)


# ------------------------------------------------------------
# 19. CREATE PREPROCESSOR
# ------------------------------------------------------------

transformers = []

if len(numeric_features) > 0:

    transformers.append(
        (
            "numeric",
            numeric_pipeline,
            numeric_features
        )
    )

if len(categorical_features) > 0:

    transformers.append(
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    )


preprocessor = ColumnTransformer(
    transformers=transformers
)


# ------------------------------------------------------------
# 20. CREATE RANDOM FOREST MODEL
# ------------------------------------------------------------

random_forest = RandomForestClassifier(
    n_estimators=200,
    criterion="gini",
    max_depth=8,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)


# ------------------------------------------------------------
# 21. CREATE COMPLETE PIPELINE
# ------------------------------------------------------------

pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            random_forest
        )
    ]
)


# ------------------------------------------------------------
# 22. TRAIN RANDOM FOREST
# ------------------------------------------------------------

print(
    "\nTraining Random Forest Classifier..."
)

pipeline.fit(
    X_train,
    y_train
)

print(
    "Random Forest training completed!"
)


# ------------------------------------------------------------
# 23. MAKE PREDICTIONS
# ------------------------------------------------------------

y_pred = pipeline.predict(
    X_test
)

y_proba = pipeline.predict_proba(
    X_test
)


# ------------------------------------------------------------
# 24. EVALUATE MODEL
# ------------------------------------------------------------

accuracy = accuracy_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred,
    average="binary",
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    average="binary",
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    average="binary",
    zero_division=0
)


print("\n========== RANDOM FOREST PERFORMANCE ==========")

print(
    f"Accuracy : {accuracy:.4f}"
)

print(
    f"Precision: {precision:.4f}"
)

print(
    f"Recall   : {recall:.4f}"
)

print(
    f"F1 Score : {f1:.4f}"
)


# ------------------------------------------------------------
# 25. ROC-AUC
# ------------------------------------------------------------

try:

    roc_auc = roc_auc_score(
        y_test,
        y_proba[:, 1]
    )

    print(
        f"ROC-AUC  : {roc_auc:.4f}"
    )

except Exception as e:

    print(
        "ROC-AUC could not be calculated:",
        e
    )


# ------------------------------------------------------------
# 26. CLASSIFICATION REPORT
# ------------------------------------------------------------

print(
    "\n========== CLASSIFICATION REPORT =========="
)

print(
    classification_report(
        y_test,
        y_pred,
        zero_division=0
    )
)


# ------------------------------------------------------------
# 27. CONFUSION MATRIX
# ------------------------------------------------------------

cm = confusion_matrix(
    y_test,
    y_pred
)

plt.figure(figsize=(7, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.title(
    "Random Forest - Confusion Matrix"
)

plt.xlabel(
    "Predicted Class"
)

plt.ylabel(
    "Actual Class"
)

plt.show()


# ------------------------------------------------------------
# 28. ROC CURVE
# ------------------------------------------------------------

try:

    fpr, tpr, thresholds = roc_curve(
        y_test,
        y_proba[:, 1]
    )

    plt.figure(figsize=(8, 6))

    plt.plot(
        fpr,
        tpr,
        label=f"Random Forest (AUC = {roc_auc:.3f})"
    )

    plt.plot(
        [0, 1],
        [0, 1],
        linestyle="--"
    )

    plt.title(
        "ROC Curve - Random Forest"
    )

    plt.xlabel(
        "False Positive Rate"
    )

    plt.ylabel(
        "True Positive Rate"
    )

    plt.legend()

    plt.show()

except Exception as e:

    print(
        "ROC curve could not be created:",
        e
    )


# ------------------------------------------------------------
# 29. GET PROCESSED FEATURE NAMES
# ------------------------------------------------------------

feature_names = (
    pipeline
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

print(
    "\nNumber of processed features:",
    len(feature_names)
)


# ------------------------------------------------------------
# 30. FEATURE IMPORTANCE
# ------------------------------------------------------------

forest_model = (
    pipeline
    .named_steps["model"]
)

importance_values = (
    forest_model.feature_importances_
)


feature_importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importance_values
})


feature_importance_df = (
    feature_importance_df
    .sort_values(
        by="Importance",
        ascending=False
    )
)


print(
    "\n========== TOP FEATURE IMPORTANCE =========="
)

display(
    feature_importance_df.head(20)
)


# ------------------------------------------------------------
# 31. FEATURE IMPORTANCE PLOT
# ------------------------------------------------------------

top_features = (
    feature_importance_df
    .head(15)
    .sort_values(
        by="Importance"
    )
)


plt.figure(figsize=(10, 7))

sns.barplot(
    data=top_features,
    x="Importance",
    y="Feature"
)

plt.title(
    "Top Features - Random Forest"
)

plt.xlabel(
    "Feature Importance"
)

plt.ylabel(
    "Feature"
)

plt.tight_layout()

plt.show()


# ------------------------------------------------------------
# 32. RANDOM FOREST INFORMATION
# ------------------------------------------------------------

print(
    "\n========== RANDOM FOREST INFORMATION =========="
)

print(
    "Number of trees:",
    forest_model.n_estimators
)

print(
    "Maximum tree depth:",
    forest_model.max_depth
)

print(
    "Number of features used per split:",
    forest_model.max_features
)


# ------------------------------------------------------------
# 33. DISPLAY ONE TREE FROM THE FOREST
# ------------------------------------------------------------

plt.figure(figsize=(25, 15))

plot_tree(
    forest_model.estimators_[0],
    feature_names=feature_names,
    class_names=[
        str(c)
        for c in forest_model.classes_
    ],
    filled=True,
    rounded=True,
    max_depth=3,
    fontsize=8
)

plt.title(
    "One Decision Tree from the Random Forest"
)

plt.show()


# ------------------------------------------------------------
# 34. ACTUAL VS PREDICTED
# ------------------------------------------------------------

comparison = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
})


print(
    "\n========== ACTUAL VS PREDICTED =========="
)

display(
    comparison.head(20)
)


# ------------------------------------------------------------
# 35. PREDICT A NEW PATIENT
# ------------------------------------------------------------

# Using an existing patient as a template
# guarantees the correct feature structure.

new_patient = X_test.iloc[[0]].copy()


print(
    "\n========== SAMPLE PATIENT =========="
)

display(
    new_patient
)


new_prediction = pipeline.predict(
    new_patient
)

new_probability = pipeline.predict_proba(
    new_patient
)


print(
    "\n========== NEW PATIENT PREDICTION =========="
)

print(
    "Predicted Heart Disease Class:",
    new_prediction[0]
)


print(
    "\nPrediction probabilities:"
)

for class_label, probability in zip(
    forest_model.classes_,
    new_probability[0]
):

    print(
        f"Class {class_label}: "
        f"{probability:.4f}"
    )


# ------------------------------------------------------------
# 36. SAVE PREDICTIONS
# ------------------------------------------------------------

prediction_results = X_test.copy()

prediction_results["Actual"] = (
    y_test.values
)

prediction_results["Predicted"] = (
    y_pred
)


prediction_file = (
    "Heart_Disease_Random_Forest_Predictions.csv"
)


prediction_results.to_csv(
    prediction_file,
    index=False
)


print(
    "\nPrediction file saved as:"
)

print(
    prediction_file
)


# ------------------------------------------------------------
# 37. DOWNLOAD PREDICTIONS
# ------------------------------------------------------------

files.download(
    prediction_file
)


# ------------------------------------------------------------
# 38. COMPARE WITH DECISION TREE
# ------------------------------------------------------------

print("\n")
print("=" * 60)

print(
    "RANDOM FOREST VS DECISION TREE"
)

print("=" * 60)

print(
    "Random Forest Accuracy:",
    round(accuracy, 4)
)

print(
    "Random Forest F1 Score:",
    round(f1, 4)
)

try:

    print(
        "Random Forest ROC-AUC:",
        round(roc_auc, 4)
    )

except:

    pass

print(
    "\nRandom Forest combines many decision trees"
)

print(
    "to improve stability and reduce overfitting."
)

print("=" * 60)


# ------------------------------------------------------------
# 39. FINAL PROJECT SUMMARY
# ------------------------------------------------------------

print("\n")

print("=" * 60)

print(
    "PROJECT 9 - FINAL SUMMARY"
)

print("=" * 60)

print(
    "Dataset shape:",
    df.shape
)

print(
    "\nTarget variable:",
    TARGET
)

print(
    "\nModel:"
)

print(
    "Random Forest Classifier"
)

print(
    "\nNumber of trees:",
    forest_model.n_estimators
)

print(
    "\nPreprocessing:"
)

print(
    "- Missing value imputation"
)

print(
    "- Numerical feature scaling"
)

print(
    "- Categorical feature encoding"
)

print(
    "\nEvaluation Metrics:"
)

print(
    f"Accuracy : {accuracy:.4f}"
)

print(
    f"Precision: {precision:.4f}"
)

print(
    f"Recall   : {recall:.4f}"
)

print(
    f"F1 Score : {f1:.4f}"
)

try:

    print(
        f"ROC-AUC  : {roc_auc:.4f}"
    )

except:

    pass


print(
    "\nThe Random Forest Ensemble uses multiple"
)

print(
    "decision trees to classify heart disease."
)

print(
    "It generally provides more robust predictions"
)

print(
    "than a single Decision Tree."
)

print("=" * 60)